# HW6 微調：分類微調、LoRA 與遺忘

> 說明：<https://github.com/chang-ye-tu/genai/blob/main/hw/hw6.md>　取材：Raschka《Build a Large Language Model (From Scratch)》第 6 章與附錄 E（套件 `llms-from-scratch`）
> 授權：本筆記本呼叫並改寫 Sebastian Raschka 的開源專案 <https://github.com/rasbt/LLMs-from-scratch>（Apache-2.0，© Sebastian Raschka）的範例程式；改寫部分（本課程的講解、問題與資料）同樣以 Apache-2.0 散布，完整授權文本見 repo 的 `LICENSES/Apache-2.0.txt`。
> 做法：**執行階段 → 變更執行階段類型 → T4 GPU**，由上而下逐格執行；看到「✍️ 請回答」就把觀察寫進該文字格。全部跑完後「檔案 → 下載 → .ipynb」上傳 iLearn，再作答實作測驗。
> **請勿更改模型名稱、版本、隨機種子與資料檔**，否則實作測驗的數值題會對不上。
> 需要 T4 GPU（兩次微調各約 2–3 分鐘）。
> 核心段落：第 0–3 節（實作測驗只出這些段落的題目）；第 4 節為選做。


In [ ]:
import hashlib
def sha256_of(path, expected=None):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    if expected is not None and digest != expected:
        raise RuntimeError(f"{path} 的 SHA-256 與課程固定版本不符：{digest[:12]}… ≠ {expected[:12]}…；請刪除檔案重新下載，或到 Q&A 回報")
    print(f"SHA-256 OK：{path} ({digest[:12]}…)")
    return digest

def fetch(path, urls, expected):
    # 依序嘗試 urls：每個網址下載後立刻驗 SHA-256，不符就刪掉換下一個來源；已存在且正確的檔案直接使用；全部來源都失敗才報錯
    import os, requests
    urls = [urls] if isinstance(urls, str) else list(urls)
    def _ok():
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest() == expected
    if os.path.exists(path):
        if _ok():
            print(f"SHA-256 OK：{path} ({expected[:12]}…)"); return path
        print(f"{path} 的 SHA-256 與課程固定版本不符，刪除後重新下載"); os.remove(path)
    for url in urls:
        try:
            with requests.get(url, timeout=600, stream=True) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
        except Exception as e:
            print("下載失敗：", url, e)
            if os.path.exists(path): os.remove(path)
            continue
        if _ok():
            print("下載自", url); print(f"SHA-256 OK：{path} ({expected[:12]}…)"); return path
        print(f"{url} 下載的內容 SHA-256 不符，改用下一個來源"); os.remove(path)
    raise RuntimeError(f"{path} 所有來源都無法取得正確的檔案（下載失敗或 SHA-256 不符）；請到 Q&A 回報")

# @title 第 0 節：安裝、資料與預訓練模型
%pip -q install --no-deps llms-from-scratch==1.0.19
%pip -q install tiktoken==0.14.0 safetensors==0.8.0
import torch, tiktoken, pandas as pd, time, requests, zipfile, os, shutil
from pathlib import Path
from torch.utils.data import DataLoader
from llms_from_scratch.ch04 import GPTModel
from llms_from_scratch.ch05 import generate, text_to_token_ids, token_ids_to_text
from llms_from_scratch.ch06 import (download_and_unzip_spam_data, create_balanced_dataset, random_split, SpamDataset,
                                    calc_accuracy_loader, train_classifier_simple, classify_review)
from llms_from_scratch.appendix_e import replace_linear_with_lora
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("裝置：", device)
import platform, importlib.metadata as _meta
def _v(p):
    try: return _meta.version(p)
    except Exception: return "missing"  # metadata 查不到時印 missing；若同一格更早的 import 已失敗，程式到不了這裡，check_submissions 會判「無版本資訊／執行錯誤」
print("VERSIONS", "python=" + platform.python_version(), "torch=" + torch.__version__, *[p + "=" + _v(p) for p in ["llms-from-scratch", "tiktoken", "safetensors"]])
tokenizer = tiktoken.get_encoding("gpt2")


## 第 1 節 資料：SMS 垃圾訊息（UCI）

5,574 則英文簡訊，標成 ham／spam。類別不平衡，先把 ham 隨機抽到和 spam 一樣多，再切 70／10／20。


In [ ]:
# 下載 UCI SMS Spam Collection（CC BY 4.0）；UCI 失敗時改抓課程 repo 內的備份（公開 repo 推送前該網址為 404，2026-09-05 確認 UCI 為 200）。每次執行都驗證 zip 與 TSV 的 SHA-256，不符會自動重下
ZIP, TSV = Path("sms_spam_collection.zip"), Path("sms_spam_collection/SMSSpamCollection.tsv")
fetch(str(ZIP), ["https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip", "https://raw.githubusercontent.com/chang-ye-tu/genai/main/hw/hw6/sms_spam_collection.zip"], "1587ea43e58e82b14ff1f5425c88e17f8496bfcdb67a583dbff9eefaf9963ce3")
if not (TSV.exists() and hashlib.sha256(TSV.read_bytes()).hexdigest() == "7d039a24a6083ed9ef0f806ebad56bbb976e3aeb8de05669173bfdc4996c239d"):  # 解壓出的 TSV 也每次驗，不符就從驗證過的 zip 重新解壓
    shutil.rmtree("sms_spam_collection", ignore_errors=True)
    with zipfile.ZipFile(ZIP) as z: z.extractall("sms_spam_collection")
    os.rename("sms_spam_collection/SMSSpamCollection", TSV)
sha256_of(str(TSV), "7d039a24a6083ed9ef0f806ebad56bbb976e3aeb8de05669173bfdc4996c239d")
df = pd.read_csv(TSV, sep="\t", header=None, names=["Label", "Text"], quoting=3)  # quoting=3 = csv.QUOTE_NONE：簡訊內有未成對的雙引號，不解讀引號才是原始檔的每一行一筆（與 UCI 目錄的筆數一致）
print("原始筆數：", len(df), "|", df["Label"].value_counts().to_dict())
balanced_df = create_balanced_dataset(df)
print("平衡後：", len(balanced_df), "|", balanced_df["Label"].value_counts().to_dict())
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})
train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
print("切分：", len(train_df), len(validation_df), len(test_df))
train_df.to_csv("train.csv", index=None); validation_df.to_csv("validation.csv", index=None); test_df.to_csv("test.csv", index=None)
torch.manual_seed(123)
train_dataset = SpamDataset(csv_file="train.csv", max_length=None, tokenizer=tokenizer)
val_dataset = SpamDataset(csv_file="validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
test_dataset = SpamDataset(csv_file="test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
print("訓練集最長訊息的 token 數（其他都補到這個長度）：", train_dataset.max_length)
BATCH = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH, num_workers=0, drop_last=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH, num_workers=0, drop_last=False)


✍️ **請回答 1-1**：為什麼要先平衡類別？如果不平衡，一個「永遠猜 ham」的分類器準確率會是多少？

（在這裡作答）


## 第 2 節 載入 GPT-2 124M，把輸出層換成 2 類

微調前先看看預訓練模型對「這是不是垃圾訊息」的直接回答（它沒學過分類）。然後把 50,257 維的輸出層換成 2 維，只訓練最後一個 block、最後的正規化與新輸出層。


In [ ]:
import hashlib
def sha256_of(path, expected=None):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    if expected is not None and digest != expected:
        raise RuntimeError(f"{path} 的 SHA-256 與課程固定版本不符：{digest[:12]}… ≠ {expected[:12]}…；請刪除檔案重新下載，或到 Q&A 回報")
    print(f"SHA-256 OK：{path} ({digest[:12]}…)")
    return digest

def fetch(path, urls, expected):
    # 依序嘗試 urls：每個網址下載後立刻驗 SHA-256，不符就刪掉換下一個來源；已存在且正確的檔案直接使用；全部來源都失敗才報錯
    import os, requests
    urls = [urls] if isinstance(urls, str) else list(urls)
    def _ok():
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest() == expected
    if os.path.exists(path):
        if _ok():
            print(f"SHA-256 OK：{path} ({expected[:12]}…)"); return path
        print(f"{path} 的 SHA-256 與課程固定版本不符，刪除後重新下載"); os.remove(path)
    for url in urls:
        try:
            with requests.get(url, timeout=600, stream=True) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
        except Exception as e:
            print("下載失敗：", url, e)
            if os.path.exists(path): os.remove(path)
            continue
        if _ok():
            print("下載自", url); print(f"SHA-256 OK：{path} ({expected[:12]}…)"); return path
        print(f"{url} 下載的內容 SHA-256 不符，改用下一個來源"); os.remove(path)
    raise RuntimeError(f"{path} 所有來源都無法取得正確的檔案（下載失敗或 SHA-256 不符）；請到 Q&A 回報")

# GPT-2 124M 預訓練權重：從 Hugging Face 下載 safetensors（548 MB），對應到我們自己實作的 GPTModel
import os, requests
from safetensors.torch import load_file
SF = "model-gpt2.safetensors"
fetch(SF, f"https://huggingface.co/openai-community/gpt2/resolve/607a30d783dfa663caf39e06633721c8d4cfcd7e/model.safetensors", "248dfc3911869ec493c76e65bf2fcf7f615828b0254c12b473182f0f81d3a707")  # 已存在且 hash 正確就不重下；不符會自動刪除重下一次
state_dict = load_file(SF)
print("張量數：", len(state_dict), "| 檔案大小 MB：", round(os.path.getsize(SF) / 1e6))

def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(right.detach().clone())

def load_hf_weights_into_gpt(gpt, p):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, p["wpe.weight"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, p["wte.weight"])
    for b in range(len(gpt.trf_blocks)):
        t = gpt.trf_blocks[b]
        qw, kw, vw = torch.chunk(p[f"h.{b}.attn.c_attn.weight"], 3, dim=-1)
        qb, kb, vb = torch.chunk(p[f"h.{b}.attn.c_attn.bias"], 3, dim=-1)
        t.att.W_query.weight = assign(t.att.W_query.weight, qw.T); t.att.W_query.bias = assign(t.att.W_query.bias, qb)
        t.att.W_key.weight = assign(t.att.W_key.weight, kw.T);     t.att.W_key.bias = assign(t.att.W_key.bias, kb)
        t.att.W_value.weight = assign(t.att.W_value.weight, vw.T); t.att.W_value.bias = assign(t.att.W_value.bias, vb)
        t.att.out_proj.weight = assign(t.att.out_proj.weight, p[f"h.{b}.attn.c_proj.weight"].T)
        t.att.out_proj.bias = assign(t.att.out_proj.bias, p[f"h.{b}.attn.c_proj.bias"])
        t.ff.layers[0].weight = assign(t.ff.layers[0].weight, p[f"h.{b}.mlp.c_fc.weight"].T)
        t.ff.layers[0].bias = assign(t.ff.layers[0].bias, p[f"h.{b}.mlp.c_fc.bias"])
        t.ff.layers[2].weight = assign(t.ff.layers[2].weight, p[f"h.{b}.mlp.c_proj.weight"].T)
        t.ff.layers[2].bias = assign(t.ff.layers[2].bias, p[f"h.{b}.mlp.c_proj.bias"])
        t.norm1.scale = assign(t.norm1.scale, p[f"h.{b}.ln_1.weight"]); t.norm1.shift = assign(t.norm1.shift, p[f"h.{b}.ln_1.bias"])
        t.norm2.scale = assign(t.norm2.scale, p[f"h.{b}.ln_2.weight"]); t.norm2.shift = assign(t.norm2.shift, p[f"h.{b}.ln_2.bias"])
    gpt.final_norm.scale = assign(gpt.final_norm.scale, p["ln_f.weight"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, p["ln_f.bias"])
    gpt.out_head.weight = assign(gpt.out_head.weight, p["wte.weight"])  # 權重共享：輸出層 = 詞嵌入

GPT2_CFG = {"vocab_size": 50257, "context_length": 1024, "emb_dim": 768, "n_heads": 12, "n_layers": 12, "drop_rate": 0.0, "qkv_bias": True}  # 與原始 GPT-2 一致：有 QKV 偏差、上下文 1024

def fresh_gpt2():
    m = GPTModel(GPT2_CFG); load_hf_weights_into_gpt(m, state_dict); return m
model = fresh_gpt2().to(device).eval()
text_1 = "Is the following text 'spam'? Answer with 'yes' or 'no': 'You are a winner you have been specially selected to receive $1000 cash or a $2000 award.'"
ids = text_to_token_ids(text_1, tokenizer).to(device)
print("微調前直接問：", repr(token_ids_to_text(generate(model, ids, max_new_tokens=23, context_size=1024, temperature=0.0), tokenizer)[len(text_1):]))


In [ ]:
torch.manual_seed(123)
model = fresh_gpt2()
for p in model.parameters(): p.requires_grad = False
model.out_head = torch.nn.Linear(in_features=GPT2_CFG["emb_dim"], out_features=2)
for p in model.trf_blocks[-1].parameters(): p.requires_grad = True
for p in model.final_norm.parameters(): p.requires_grad = True
model.to(device)
print("可訓練參數（最後一個 block + 正規化 + 輸出層）：", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}", "/ 全部", f"{sum(p.numel() for p in model.parameters()):,}")
print("微調前 測試集準確率：", f"{calc_accuracy_loader(test_loader, model, device, num_batches=10):.2%}")


In [ ]:
NUM_EPOCHS = 5
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
t0 = time.time()
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device, num_epochs=NUM_EPOCHS, eval_freq=50, eval_iter=5)
print(f"訓練時間：{(time.time()-t0)/60:.1f} 分")
print("訓練集準確率：", f"{calc_accuracy_loader(train_loader, model, device):.2%}", "| 驗證集：", f"{calc_accuracy_loader(val_loader, model, device):.2%}", "| 測試集：", f"{calc_accuracy_loader(test_loader, model, device):.2%}")


In [ ]:
for t in ["You are a winner you have been specially selected to receive $1000 cash or a $2000 award.",
          "Hey, just wanted to check if we're still on for dinner tonight? Let me know!",
          "恭喜！您已獲得 iPhone 抽獎資格，請點擊連結領取。"]:
    print(classify_review(t, model, tokenizer, device, max_length=train_dataset.max_length), "←", t[:60])


✍️ **請回答 2-1**：測試集準確率多少？只訓練約 7 百萬個參數就能做到，說明預訓練模型已經學到了什麼？中文簡訊為什麼判斷得不可靠？

（在這裡作答）


## 第 3 節 LoRA：凍結原本的權重，另外加上低秩的小矩陣

第 10 單元講的 LoRA：每個線性層 `Linear`（把 in 維映到 out 維）旁邊加上 A（in×r）與 B（r×out），輸出變成 `Linear(x) + α·(x @ A @ B)`（本套件 1.0.19 的實作用 α 直接縮放，沒有除以 r；原論文是 α/r）。預訓練權重全部凍結，可訓練的是所有 A、B 加上新的 2 類分類輸出層；B 初始為 0，所以訓練起點就是第 2 節「換上新分類頭」的那個模型（增量為零），不是原本的語言模型。比較可訓練參數量與準確率。


In [ ]:
torch.manual_seed(123)
model_lora = fresh_gpt2()
for p in model_lora.parameters(): p.requires_grad = False
model_lora.out_head = torch.nn.Linear(in_features=GPT2_CFG["emb_dim"], out_features=2)
LORA_RANK, LORA_ALPHA = 16, 16
replace_linear_with_lora(model_lora, rank=LORA_RANK, alpha=LORA_ALPHA)
model_lora.to(device)
n_train = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"LoRA r={LORA_RANK} 可訓練參數：{n_train:,}（所有 A、B 矩陣 + 新的 2 類輸出層 1,538 個）")
print(model_lora.trf_blocks[0].att.W_query)


In [ ]:
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model_lora.parameters(), lr=5e-5, weight_decay=0.1)
t0 = time.time()
_ = train_classifier_simple(model_lora, train_loader, val_loader, optimizer, device, num_epochs=NUM_EPOCHS, eval_freq=50, eval_iter=5)
print(f"訓練時間：{(time.time()-t0)/60:.1f} 分")
print("LoRA 測試集準確率：", f"{calc_accuracy_loader(test_loader, model_lora, device):.2%}")


✍️ **請回答 3-1**：LoRA 的可訓練參數是第 2 節做法的幾分之一？準確率差多少？r 從 16 改成 8，A、B 的參數會減半但新輸出層的 1,538 個不變——總數會變成多少（可以改 LORA_RANK 重跑第一格驗證）？

（在這裡作答）


## 第 4 節（選做）遺忘：微調後的模型還會「說話」嗎

分類微調把輸出層換掉了，模型只會輸出 2 個數字，原本的文字接龍能力當然不見了；即使不換輸出層，全參數微調也可能讓模型忘記原有能力（第 10 單元的災難性遺忘）。這裡用一個對照實驗感受：把分類模型微調過的最後一層放回完整的 GPT-2，看它還能不能接龍。


In [ ]:
restored = fresh_gpt2().to(device).eval()
restored.trf_blocks[-1].load_state_dict(model.trf_blocks[-1].state_dict())   # 拿回微調過的最後一層
restored.final_norm.load_state_dict(model.final_norm.state_dict())
for m_, name in [(fresh_gpt2().to(device).eval(), "原始 GPT-2"), (restored, "換回輸出層的微調模型")]:
    idx = text_to_token_ids("Every effort moves", tokenizer).to(device)
    print(f"[{name}]", repr(token_ids_to_text(generate(m_, idx, max_new_tokens=20, context_size=1024, temperature=0.0), tokenizer)))


✍️ **請回答 4-1**：微調過最後一層的模型接龍還通順嗎？只動一層就有影響，全參數微調會如何？第 10 單元提到哪些方法可以減少遺忘（LoRA、experience replay、模型合併）？

（在這裡作答）


## ✍️ AI 使用聲明（必填）

| 項目 | 內容 |
|------|------|
| 使用的工具 | （例如：ChatGPT 免費版、Colab 內建 Gemini） |
| 用在哪些工作 | （例如：解釋錯誤訊息、幫我看懂某一格程式） |
| 我自己完成的部分 | （例如：全部執行、所有 ✍️ 回答） |
| 我如何驗證 AI 的說法 | （例如：實際執行、對照投影片） |

姓名／學號：
